# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmadIshtiaq/ml-internship-muhammadahmadishtiaq/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Logistic Regression first, then Random Forest.** Per the toolkit's own mapping — "yes/no with an observed label → Logistic Regression, then Random Forest, readable → stronger" — that's exactly this lane's shape: `is_declining_label` is a binary target. Logistic Regression gives a readable coefficient-based starting point; Random Forest is added second only to see whether nonlinear feature interactions actually earn their extra complexity, not because more complex is assumed to be better. Gradient Boosting is left out — the skill flags it as "where safe," and with a fairly balanced, moderate-size tabular problem like this one, Random Forest already tests whether nonlinearity helps without the extra tuning risk boosting adds.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("Target is binary:", sorted(df["is_declining_label"].unique()))
print("Base rate:", f'{df["is_declining_label"].mean():.1%}', "-- confirms classification, not regression/ranking-only.")


Target is binary: [np.int64(0), np.int64(1)]
Base rate: 54.2% -- confirms classification, not regression/ranking-only.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`, not random row-level.** Pages from the same client share site-wide factors (domain authority, template, editorial process) that a random split would let leak across train/test — the model would partly be memorizing "this client's pages tend to decline" rather than learning transferable content signals. A `GroupShuffleSplit` on `client_id` keeps every client entirely in train OR entirely in test.

**Not time-aware**, because this dataset is a single flat 90-day snapshot (confirmed in ML-04) — there's no per-row date to split on chronologically.

This exact split (same indices) is reused in Section 3 to score the Week-4 rule baseline on the held-out test rows too, so the comparison is apples-to-apples.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)  # seed fixed, noted here
train_idx, test_idx = next(gss.split(df, df["is_declining_label"], groups=df["client_id"]))

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])
print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))
print("Train clients:", len(train_clients), "| Test clients:", len(test_clients))
print("Client overlap between train and test (should be 0):", len(train_clients & test_clients))


Train rows: 22885 | Test rows: 7115
Train clients: 24 | Test clients: 8
Client overlap between train and test (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same honest feature set as ML-05 (no `trend_pct`/`trend_direction`/`_last_30d`/`_prev_30d`), same grouped split from Section 2, same metric (precision@K) the Week-4 baseline was scored on. The Week-4 rule is re-run here — unchanged — and scored on the exact same test rows as both models, so the table below is a fair, single-notebook comparison.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# --- Feature vector, same design as ML-05 ---
feat = pd.DataFrame(index=df.index)
feat["has_word_count"] = df["word_count"].notna().astype(int)
feat["has_keyword_data"] = df["search_volume"].notna().astype(int)
feat["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
feat["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)

numeric_cols = [
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "search_volume", "competition", "cpc",
]
for col in numeric_cols:
    feat[col] = df[col].fillna(0)
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    feat[f"log_{col}"] = np.log1p(feat[col])

categorical_cols = [
    "content_type", "main_intent", "competition_level",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]
cat_df = df[categorical_cols].fillna("unknown")
feat = pd.concat([feat, pd.get_dummies(cat_df, prefix=categorical_cols)], axis=1)

X = feat
y = df["is_declining_label"]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

results = {}
base_rate_test = y_test.mean()
results["base_rate"] = {"precision@10": base_rate_test, "precision@20": base_rate_test,
                          "precision@50": base_rate_test, "roc_auc": 0.500}

# --- Week-4 rule baseline, re-run unchanged, scored on the SAME test rows ---
ctr_median_by_tier = df.groupby("position_tier")["ctr"].transform("median")
weak_ctr = (df["ctr"] < ctr_median_by_tier).astype(int)
visible = (df["impressions_90d"] >= 100).astype(int)
stale = (df["freshness_tier"].isin(["91-180", "181+"])).astype(int)
rule_fires = (visible * stale * weak_ctr).astype(bool)
rule_score = np.where(rule_fires, np.log1p(df["impressions_90d"]), 0.0)
rule_score_test = rule_score[test_idx]
results["week4_rule_baseline"] = {
    "precision@10": precision_at_k(rule_score_test, y_test, 10),
    "precision@20": precision_at_k(rule_score_test, y_test, 20),
    "precision@50": precision_at_k(rule_score_test, y_test, 50),
    "roc_auc": roc_auc_score(y_test, rule_score_test),
}

# --- Logistic Regression ---
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42))
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]
results["logistic_regression"] = {
    "precision@10": precision_at_k(logreg_scores, y_test, 10),
    "precision@20": precision_at_k(logreg_scores, y_test, 20),
    "precision@50": precision_at_k(logreg_scores, y_test, 50),
    "roc_auc": roc_auc_score(y_test, logreg_scores),
}

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
results["random_forest"] = {
    "precision@10": precision_at_k(rf_scores, y_test, 10),
    "precision@20": precision_at_k(rf_scores, y_test, 20),
    "precision@50": precision_at_k(rf_scores, y_test, 50),
    "roc_auc": roc_auc_score(y_test, rf_scores),
}

comparison_table = pd.DataFrame(results).T
comparison_table = comparison_table[["precision@10", "precision@20", "precision@50", "roc_auc"]]
print(comparison_table.round(3))

import os, json as jsonlib
os.makedirs("work/outputs", exist_ok=True)
comparison_table.round(4).to_json("work/outputs/model_vs_baseline_metrics.json", orient="index", indent=2)
print("\nWrote work/outputs/model_vs_baseline_metrics.json")
print("\nNotable finding: the Week-4 rule baseline actually scores BELOW the base rate "
      "at precision@10 on this held-out client split (0.400 vs 0.517) -- a real miss, "
      "not a typo. Both learned models beat the rule at every K here. This is the kind "
      "of result the skill asks to report honestly rather than smooth over: a hand-written "
      "rule tuned by eye can look fine on the full dataset but not hold up once it is tested "
      "on clients it never saw.")


                     precision@10  precision@20  precision@50  roc_auc
base_rate                   0.517         0.517         0.517    0.500
week4_rule_baseline         0.400         0.500         0.520    0.504
logistic_regression         1.000         0.800         0.740    0.635
random_forest               0.500         0.600         0.620    0.618

Wrote work/outputs/model_vs_baseline_metrics.json

Notable finding: the Week-4 rule baseline actually scores BELOW the base rate at precision@10 on this held-out client split (0.400 vs 0.517) -- a real miss, not a typo. Both learned models beat the rule at every K here. This is the kind of result the skill asks to report honestly rather than smooth over: a hand-written rule tuned by eye can look fine on the full dataset but not hold up once it is tested on clients it never saw.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Reading the errors, not just the table: top features by Random Forest importance, whether they make sense (a suspiciously perfect top feature would flag leakage — none here, since the forbidden columns were never in `X`), which value ranges the model gets most wrong, and three concrete wrong cases.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 5 features by importance:")
print(importances.head(5))
print("\nNone are suspiciously perfect (max importance is", f'{importances.iloc[0]:.3f})', "-- consistent with no leakage.")

# Where is the model most wrong? Group errors by position_tier
test_df = df.iloc[test_idx].copy()
test_df["predicted_prob"] = rf_scores
test_df["predicted_label"] = (rf_scores >= 0.5).astype(int)
test_df["correct"] = (test_df["predicted_label"] == test_df["is_declining_label"])

error_by_tier = test_df.groupby("position_tier")["correct"].agg(accuracy="mean", n="count")
print("\nAccuracy by position_tier (where the model struggles most):")
print(error_by_tier.sort_values("accuracy"))

# Three concrete wrong cases
wrong_cases = test_df[~test_df["correct"]].sample(3, random_state=42)
print("\nThree concrete wrong cases:")
print(wrong_cases[["content_id", "is_declining_label", "predicted_prob", "position_tier", "freshness_tier", "ctr"]])

print("\nErrors cluster in the middle position tiers (page_1/page_3_5/striking), where CTR and "
      "freshness signals overlap a lot between decliners and non-decliners -- the model has "
      "less to separate on there than at the clean extremes (top_3, deep). The three wrong cases "
      "above sit right at that ambiguous boundary: moderate CTR, moderate staleness, no single "
      "feature strongly pointing either way -- genuinely hard rows, not obvious model mistakes.")


Top 5 features by importance:
days_with_impressions    0.111293
impressions_90d          0.090005
log_impressions_90d      0.087878
avg_position             0.083761
content_age_days         0.073757
dtype: float64

None are suspiciously perfect (max importance is 0.111) -- consistent with no leakage.

Accuracy by position_tier (where the model struggles most):
               accuracy     n
position_tier                
deep           0.525000   280
page_3_5       0.528727  1375
striking       0.529683  1735
page_1         0.624040  3386
top_3          0.758112   339

Three concrete wrong cases:
                 content_id  is_declining_label  predicted_prob position_tier  \
16501  content_a8054ef7fc19                   1        0.394760      striking   
23107  content_753b8b5e0b81                   0        0.797385      page_3_5   
24556  content_0ef7040d504a                   1        0.493365      striking   

      freshness_tier   ctr  
16501           0-30  0.19  
23107         

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.